In [1]:
import os
import gc
import math
import time
import torch
import pickle
import numpy as np

import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.autograd as autograd

import torch.nn.functional as functional
import torch.optim.lr_scheduler as lr_scheduler

from utils import *
os.makedirs('./phi4_8x8', exist_ok=True)
torch.set_default_dtype(torch.float32) 
_ = torch.manual_seed(0)

In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
if torch.cuda.is_available():
    torch_device = 'cuda'
else:
    torch_device = 'cpu'
print(f"TORCH DEVICE: {torch_device}")

TORCH DEVICE: cuda


In [4]:
class Score(nn.Module):
    def __init__(self,input_shape,hidden_dim):
        super().__init__()
        self.input_shape = input_shape
        self.hidden_dim  = hidden_dim
        self.net = nn.Sequential(
            nn.Conv2d(self.input_shape[0], self.hidden_dim, kernel_size = 3, stride = 1),
            nn.Tanh(),
            nn.CircularPad2d(1),
            nn.Conv2d(self.hidden_dim, self.hidden_dim, kernel_size = 3, stride = 1),
            nn.Tanh(),
            nn.CircularPad2d(1),
            nn.Conv2d(self.hidden_dim, self.hidden_dim*2, kernel_size = 3, stride = 1),
            nn.Tanh(),
            nn.CircularPad2d(1),
            nn.Conv2d(self.hidden_dim*2, self.hidden_dim*2, kernel_size = 3, stride = 1),
            nn.Tanh(),
            nn.CircularPad2d(1),
            nn.Conv2d(self.hidden_dim*2, self.hidden_dim, kernel_size = 3, stride = 1),
            nn.Tanh(),
            nn.CircularPad2d(1),
            nn.Conv2d(self.hidden_dim, self.input_shape[0], kernel_size = 3, stride = 1),
            nn.CircularPad2d(1),
        )


    def forward(self, x):
        score = self.net(x)
        return score

In [5]:
def jacobian(f, x):
    """
    Computes the Jacobian of the function f with respect to the input x.
    
    Args:
    f (function): The function for which we want the Jacobian.
    x (Tensor): The input tensor for which we compute the Jacobian. Expected shape [B, 1, M, N].
    
    Returns:
    Tensor: The Jacobian of shape [B, M', N', M * N], where B is the batch size, 
            M' and N' are the output spatial dimensions, and M * N is the input dimension.
    """
    B, C, M, N = x.shape  # B = batch size, C = number of channels (1), M and N are spatial dims
    y = f(x)  # f(x) is expected to produce an output of shape [B, 1, M', N']
    M_prime, N_prime = y.shape[2], y.shape[3]  # M' and N' are the output spatial dimensions
    
    jacobian = []
    for m in range(M_prime):
        for n in range(N_prime):
            # Create a one-hot vector for the (m, n)-th component of y
            v = torch.zeros_like(y)
            v[:, 0, m, n] = 1.0  # Set the (m, n)-th element in the 1st channel of output
            
            # Compute the gradient of y with respect to x (dy/dx)
            dy_dx = autograd.grad(y, x, grad_outputs=v, retain_graph=True, create_graph=True, allow_unused=True)[0]
            
            jacobian.append(dy_dx.view(B, -1).unsqueeze(2))  # Shape [B, M * N, 1] for each component

    # Stack all Jacobian components to get a tensor of shape [B, M', N', M * N]
    jacobian = torch.cat(jacobian, dim=2)  # Shape [B, M', N', M * N]
    
    return jacobian


In [6]:
def score_matching(model, samples):
    samples.requires_grad_(True)
    logp = model(samples)
    norm_jacobian = torch.norm(logp, dim=(2,3)) ** 2 / 2.
    # print(norm_jacobian.shape)
    jacob_mat = jacobian(model, samples)
    # print(jacob_mat.shape)
    trace_jacobian = torch.diagonal(jacob_mat, dim1=-2, dim2=-1).sum(-1)
    # print(trace_jacobian.shape)
    loss = (trace_jacobian + norm_jacobian.squeeze(-1)).mean(-1)
    return loss

In [7]:
dim = 64
data_filename = "phi4_8x8_data_for_mass_1_alongwith_neglogp.pkl"
f = open(data_filename, 'rb')
dataset = pickle.load(f, encoding="latin1")
dataset = torch.tensor(dataset[0] , dtype = torch.float32 , device = torch_device).reshape((-1,1,8,8))

In [8]:
model = Score((1,8,8),256).to(torch_device)

In [9]:
batch_size = 1
no_training_samples = 100000

In [10]:
train_dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle=True)


In [11]:
PATH = "./phi4_8x8/score_estimation_model_mass_1_lambda_4.pt"
optimizer = optim.Adam(model.parameters(),lr= 1.e-5)


In [13]:
#For resuming the checkpoint:
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [14]:
score = []
for x in train_dataloader:
    score.append(-model(x).cpu().detach().numpy())

In [15]:
score = np.array(score).reshape((-1,1,8,8))

In [16]:
score.shape

(10000, 1, 8, 8)

In [17]:
array = [dataset.cpu().detach().numpy(),score]

In [19]:
output = open("phi4_8x8_data_for_mass_1_with_score_obtained_through_score_model.pkl", 'wb')
pickle.dump(array, output)
output.close()

In [18]:
dataset.shape

torch.Size([10000, 1, 8, 8])